<a href="https://colab.research.google.com/github/bandi137/w12-trendfigyelo/blob/main/W12_trendfigyelo_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# W12 – Kutatási trendfigyelő RAG-gal

**Mit csinál a notebook:** letölti az arXiv egy kategóriájának absztraktjait havi bontásban, vektorokká alakítja őket,
kérdésekre forrásmegjelöléssel válaszol (RAG), és kimutatja a felfutó témákat.

A kód a repó `trendfigyelo/` mappájában van; ez a notebook csak beállít, meghív és megjelenít.
**Kódfrissítés után futtasd újra az első cellát, majd az alatta lévőket** (*Runtime → Run after*) – az első cella lehúzza és újratölti a legfrissebb kódot.

**Indítás előtt (egyszer):**
1. *Runtime → Change runtime type → T4 GPU*
2. Bal oldalt a 🔑 *Secrets* fülön legyen egy `GEMINI_API_KEY` nevű titok (kulcs: https://aistudio.google.com/apikey), notebook-hozzáféréssel.
3. *Runtime → Run all*. Az első futás (letöltés + vektorizálás) kb. 15–25 perc, utána a Drive-ról töltődik minden.

## 0. Kód frissítése a GitHubról

In [ ]:
REPO_URL = "https://github.com/bandi137/w12-trendfigyelo"
!if [ -d /content/repo ]; then git -C /content/repo pull -q; else git clone -q {REPO_URL} /content/repo; fi
!git -C /content/repo log -1 --format="Kódverzió: %h – %s (%cr)"
!pip install -q -r /content/repo/requirements.txt
import sys
if "/content/repo" not in sys.path:
    sys.path.insert(0, "/content/repo")
# A korábban betöltött modulok törlése, hogy a git pull után a friss kód töltődjön be
# (az IPython autoreload a Colab Pythonjával nem működik: "No module named 'imp'")
for name in [m for m in sys.modules if m == "trendfigyelo" or m.startswith("trendfigyelo.")]:
    del sys.modules[name]
import trendfigyelo as tf

## 1. Beállítások

In [ ]:
CATEGORY   = "cs.CL"        # arXiv kategória
MONTHS     = 12             # hány hónapra visszamenőleg
PER_MONTH  = 400            # cikk / hónap (12 x 400 = 4800 absztrakt)
EMB_MODEL  = "intfloat/multilingual-e5-base"   # többnyelvű -> magyar kérdés is működik
LLM_MODEL  = "gemini-3.5-flash-lite"           # ha 404-et ad, nézd meg az aktuális nevet az AI Studióban
TOP_K      = 5              # ennyi cikket kap meg a nyelvi modell
N_TOPICS   = 25             # témák száma a trendelemzéshez
N_SHOW     = 8              # ennyi téma kerül a táblázatba

import os
from google.colab import drive, userdata
from google import genai
drive.mount("/content/drive")
DATA_DIR = f"/content/drive/MyDrive/W12_trendfigyelo/{CATEGORY.replace('.', '_')}"
os.makedirs(DATA_DIR, exist_ok=True)
llm = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
print("Adatmappa:", DATA_DIR)

## 2. Adatgyűjtés az arXivról (egyszer fut, utána a Drive-ról töltődik)

In [ ]:
df = tf.load_or_download(DATA_DIR, CATEGORY, MONTHS, PER_MONTH)
df.head()

## 3. Vektorizálás (egyszer fut, utána a Drive-ról töltődik)

In [ ]:
embedder = tf.Embedder(EMB_MODEL)
emb = tf.load_or_embed(df, embedder, DATA_DIR)
emb.shape

## 4. Kérdés–válasz forrásmegjelöléssel (RAG)

In [ ]:
rag = tf.RAG(df, emb, embedder.query, llm, LLM_MODEL, top_k=TOP_K)
rag.ask("Milyen módszereket javasolnak mostanában a nagy nyelvi modellek hallucinációinak csökkentésére?");

In [ ]:
rag.ask("What are recent approaches to evaluating LLM agents?");

## 5. Trendek: témák, nevek, havi részarány

In [ ]:
df["topic"], keywords = tf.find_topics(emb, (df.title + ". " + df.abstract).values, n_topics=N_TOPICS)
share, growth = tf.topic_trends(df.month, df.topic)
topic_ids = [int(t) for t in growth.index]          # növekedés szerint csökkenő sorrendben

names = tf.name_topics(llm, LLM_MODEL, df, keywords, topic_ids, CATEGORY, DATA_DIR)
table = tf.trend_table(df, topic_ids, names, keywords, growth)
table.to_csv(f"{DATA_DIR}/trendek.csv", index=False)
display(tf.style_table(table, N_SHOW))

In [ ]:
tf.plot_trends(share, growth, names, topic_ids, CATEGORY, path=f"{DATA_DIR}/trendek.png");

In [ ]:
print(tf.summarize_trends(llm, LLM_MODEL, names, growth, topic_ids, CATEGORY, n=N_SHOW))

## 6. Kiértékelés

Töltsétek fel a listát 15–20 kérdéssel. Az `expected` mezőbe azoknak a cikkeknek az azonosítója kerül, amelyeknek
a találatok között kell lenniük (a `rag.retrieve("...", k=10)` és a `df` böngészésével lehet megkeresni őket).

In [ ]:
EVAL_QUESTIONS = [
    # {"q": "Hogyan lehet csökkenteni a hallucinációt RAG-rendszerekben?", "expected": ["2503.01234"]},
]
if EVAL_QUESTIONS:
    eval_df, summary = tf.evaluate(EVAL_QUESTIONS, lambda q, k: rag.retrieve(q, k=k))
    eval_df.to_csv(f"{DATA_DIR}/eval.csv", index=False)
    display(eval_df); print(summary)

In [ ]:
# Kérdés–válasz napló a dokumentációhoz (a kézi 1-5 pontozást ebbe a táblába lehet beírni)
rag.save_log(f"{DATA_DIR}/qa_naplo.csv")